# 11 Grad-CAM generation

Robust checkpoint selection, explicit Grad-CAM target mode, and heatmaps saved in model-input coordinates.

In [1]:
from pathlib import Path
import os, json, shutil, zipfile, warnings, random
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt


# ============================================================
# 11. GRADCAM GENERATION — FINAL PUBLICATION-READY CELL
# ============================================================

SEED = int(os.environ.get("THERMO_SEED", 42))
random.seed(SEED)
np.random.seed(SEED)

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".webp"}


# -----------------------------
# Project paths
# -----------------------------

_candidate_bases = [
    Path("/content"),
    Path("/mnt/data"),
    Path.cwd(),
    Path("/tmp"),
]

def _base_is_usable(p):
    try:
        return p.exists() and os.access(p, os.W_OK)
    except Exception:
        return False

BASE_DIR = Path(
    os.environ.get(
        "THERMO_BASE_DIR",
        str(next((p for p in _candidate_bases if _base_is_usable(p)), Path("/tmp")))
    )
)

PROJECT_NAME = os.environ.get("THERMO_PROJECT_NAME", "project_thermography_equine")
PROJECT_ROOT = BASE_DIR / PROJECT_NAME

DATA_ROOT = PROJECT_ROOT / "data"
SPLIT_DATA_DIR = DATA_ROOT / "dataset_split"
PROCESSED_DIR = DATA_ROOT / "processed"
CLEAN_IMAGE_DIR = PROCESSED_DIR / "clean_images"

OUTPUT_ROOT = PROJECT_ROOT / "outputs"
CONFIG_DIR = OUTPUT_ROOT / "config"
REPORTS_DIR = OUTPUT_ROOT / "reports"
MODELS_DIR = OUTPUT_ROOT / "models"
GRADCAM_DIR = OUTPUT_ROOT / "gradcam"

for d in [
    PROJECT_ROOT,
    DATA_ROOT,
    SPLIT_DATA_DIR,
    PROCESSED_DIR,
    CLEAN_IMAGE_DIR,
    OUTPUT_ROOT,
    CONFIG_DIR,
    REPORTS_DIR,
    MODELS_DIR,
    GRADCAM_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)


# -----------------------------
# Helpers
# -----------------------------

def count_images(root):
    root = Path(root)
    if not root.exists():
        return 0
    return sum(
        1 for p in root.rglob("*")
        if p.is_file()
        and p.suffix.lower() in IMAGE_EXTS
        and ".ipynb_checkpoints" not in p.parts
    )


def read_csv_required(path, required):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Required CSV not found: {path}")

    df = pd.read_csv(path)

    missing = [c for c in required if c not in df.columns]
    if missing:
        raise KeyError(f"{path} missing required columns: {missing}")

    return df


def find_first(patterns, roots):
    hits = []

    for root in roots:
        root = Path(root)
        if not root.exists():
            continue

        for pat in patterns:
            hits.extend([
                x for x in root.rglob(pat)
                if x.is_file()
                and ".ipynb_checkpoints" not in x.parts
            ])

    if not hits:
        return None

    return sorted(hits, key=lambda x: x.stat().st_mtime, reverse=True)[0]


def safe_copy(src, dst):
    src = Path(src)
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)

    if src.exists() and src.resolve() != dst.resolve():
        shutil.copy2(src, dst)

    return dst


def unzip_archives_if_needed():
    """
    Grad-CAM cannot read images inside ZIP files.
    This block extracts image ZIP archives into the expected project folders.
    """

    search_dirs = [
        PROJECT_ROOT,
        DATA_ROOT,
        PROCESSED_DIR,
        CLEAN_IMAGE_DIR,
        SPLIT_DATA_DIR,
        Path("/content"),
        Path("/mnt/data"),
    ]

    zip_files = []

    for root in search_dirs:
        root = Path(root)
        if not root.exists():
            continue

        zip_files.extend([
            p for p in root.rglob("*.zip")
            if p.is_file()
            and ".ipynb_checkpoints" not in p.parts
        ])

    zip_files = sorted(set(zip_files))

    before_clean = count_images(CLEAN_IMAGE_DIR)
    before_split = count_images(SPLIT_DATA_DIR)

    extracted = []
    skipped = []

    for zpath in zip_files:
        lname = zpath.name.lower()

        if "clean" in lname or "224" in lname or "processed" in lname:
            out_dir = CLEAN_IMAGE_DIR
        elif "split" in lname or "dataset" in lname:
            out_dir = SPLIT_DATA_DIR
        elif "image" in lname or "img" in lname:
            out_dir = CLEAN_IMAGE_DIR
        else:
            out_dir = zpath.parent

        try:
            with zipfile.ZipFile(zpath, "r") as z:
                members = [
                    m for m in z.namelist()
                    if Path(m).suffix.lower() in IMAGE_EXTS
                    and "__MACOSX" not in m
                ]

                if not members:
                    skipped.append({
                        "zip": str(zpath),
                        "reason": "no image files detected"
                    })
                    continue

                z.extractall(out_dir)
                extracted.append({
                    "zip": str(zpath),
                    "to": str(out_dir),
                    "n_image_members": len(members)
                })

        except Exception as e:
            skipped.append({
                "zip": str(zpath),
                "reason": repr(e)
            })

    after_clean = count_images(CLEAN_IMAGE_DIR)
    after_split = count_images(SPLIT_DATA_DIR)

    unzip_report = {
        "auto_unzip_executed": True,
        "n_zip_files_found": len(zip_files),
        "n_archives_extracted": len(extracted),
        "n_archives_skipped": len(skipped),
        "clean_images_before": before_clean,
        "clean_images_after": after_clean,
        "dataset_split_images_before": before_split,
        "dataset_split_images_after": after_split,
        "extracted_archives": extracted,
        "skipped_archives": skipped,
    }

    (REPORTS_DIR / "gradcam_auto_unzip_report.json").write_text(
        json.dumps(unzip_report, indent=2),
        encoding="utf-8"
    )

    print(json.dumps(unzip_report, indent=2))

    return unzip_report


def build_image_index(roots):
    """
    Builds filename-based index for robust image path recovery.
    """
    index = {}

    for root in roots:
        root = Path(root)
        if not root.exists():
            continue

        for p in root.rglob("*"):
            if (
                p.is_file()
                and p.suffix.lower() in IMAGE_EXTS
                and ".ipynb_checkpoints" not in p.parts
            ):
                index.setdefault(p.name, []).append(p)
                index.setdefault(p.stem, []).append(p)

    return index


# -----------------------------
# Auto-recover upstream outputs
# -----------------------------

unzip_archives_if_needed()

for target, pats, outdir in [
    ("cnn_model_predictions.csv", ["cnn_model_predictions*.csv"], CONFIG_DIR),
    ("cnn_model_record.json", ["cnn_model_record*.json"], CONFIG_DIR),
]:
    if not (outdir / target).exists():
        f = find_first(
            pats,
            [
                PROJECT_ROOT,
                OUTPUT_ROOT,
                CONFIG_DIR,
                REPORTS_DIR,
                BASE_DIR,
                Path("/mnt/data"),
                Path("/content"),
            ]
        )
        if f is not None:
            safe_copy(f, outdir / target)

for pat in ["cnn_*best*.pt", "cnn_*.pt", "*.pth"]:
    f = find_first(
        [pat],
        [
            MODELS_DIR,
            OUTPUT_ROOT,
            PROJECT_ROOT,
            BASE_DIR,
            Path("/mnt/data"),
            Path("/content"),
        ]
    )
    if f is not None:
        safe_copy(f, MODELS_DIR / f.name)


# -----------------------------
# Load predictions and model record
# -----------------------------

pred = read_csv_required(
    CONFIG_DIR / "cnn_model_predictions.csv",
    [
        "image_name",
        "split",
        "label_binary",
        "cnn_probability_pathological",
        "cnn_predicted_label_binary",
    ]
)

record_path = CONFIG_DIR / "cnn_model_record.json"
record = json.loads(record_path.read_text(encoding="utf-8")) if record_path.exists() else {}

if record.get("test_set_used_for_model_selection") is True:
    raise AssertionError(
        "cnn_model_record.json indicates that the test set was used for model selection."
    )


# -----------------------------
# Resolve image paths
# -----------------------------

image_roots = [
    CLEAN_IMAGE_DIR,
    SPLIT_DATA_DIR,
    PROCESSED_DIR,
    DATA_ROOT,
    PROJECT_ROOT,
    BASE_DIR,
    Path("/content"),
    Path("/mnt/data"),
]

image_index = build_image_index(image_roots)

def resolve_image_path(row):
    candidates = []

    for col in [
        "clean_image_path",
        "feature_image_path",
        "image_path",
        "resolved_image_path",
        "processed_image_path",
        "preprocessed_image_path",
    ]:
        if col in row and pd.notna(row[col]):
            raw = str(row[col]).strip()
            if raw and raw.lower() not in ["nan", "none"]:
                p = Path(raw)
                candidates.extend([
                    p,
                    PROJECT_ROOT / p,
                    DATA_ROOT / p,
                    PROCESSED_DIR / p,
                    CLEAN_IMAGE_DIR / p.name,
                    SPLIT_DATA_DIR / p.name,
                ])

    if "relative_image_path" in row and pd.notna(row["relative_image_path"]):
        rel = Path(str(row["relative_image_path"]).strip())
        candidates.extend([
            CLEAN_IMAGE_DIR / rel,
            SPLIT_DATA_DIR / rel,
            PROCESSED_DIR / rel,
            DATA_ROOT / rel,
            PROJECT_ROOT / rel,
        ])

    image_name = str(row.get("image_name", "")).strip()
    if image_name and image_name.lower() not in ["nan", "none", ""]:
        name = Path(image_name).name
        stem = Path(image_name).stem

        if name in image_index:
            candidates.extend(image_index[name])

        if stem in image_index:
            candidates.extend(image_index[stem])

    seen = set()

    for c in candidates:
        c = Path(c)

        try:
            key = str(c.resolve()) if c.exists() else str(c)
        except Exception:
            key = str(c)

        if key in seen:
            continue
        seen.add(key)

        if (
            c.exists()
            and c.is_file()
            and c.suffix.lower() in IMAGE_EXTS
            and ".ipynb_checkpoints" not in c.parts
        ):
            return str(c)

    return None


pred["resolved_image_path"] = pred.apply(resolve_image_path, axis=1)
pred.to_csv(CONFIG_DIR / "gradcam_input_manifest.csv", index=False)

unresolved = pred[pred["resolved_image_path"].isna()].copy()

path_resolution_report = {
    "n_predictions": int(len(pred)),
    "n_resolved_images": int(pred["resolved_image_path"].notna().sum()),
    "n_unresolved_images": int(pred["resolved_image_path"].isna().sum()),
    "n_images_in_clean_image_dir": count_images(CLEAN_IMAGE_DIR),
    "n_images_in_dataset_split": count_images(SPLIT_DATA_DIR),
    "example_unresolved_image_names": (
        unresolved["image_name"].head(30).tolist()
        if "image_name" in unresolved.columns
        else []
    ),
}

(REPORTS_DIR / "gradcam_path_resolution_report.json").write_text(
    json.dumps(path_resolution_report, indent=2),
    encoding="utf-8"
)

print(json.dumps(path_resolution_report, indent=2))


# -----------------------------
# Resolve checkpoint
# -----------------------------

checkpoint_path = None

if record.get("checkpoint_path"):
    checkpoint_path = Path(record["checkpoint_path"])

    if not checkpoint_path.exists():
        checkpoint_path = MODELS_DIR / checkpoint_path.name

if checkpoint_path is None or not checkpoint_path.exists():
    checkpoint_path = find_first(
        ["cnn_*best*.pt", "cnn_*.pt", "*.pth"],
        [
            MODELS_DIR,
            OUTPUT_ROOT,
            PROJECT_ROOT,
            BASE_DIR,
            Path("/mnt/data"),
            Path("/content"),
        ]
    )

can_attempt = (
    checkpoint_path is not None
    and Path(checkpoint_path).exists()
    and pred["resolved_image_path"].notna().any()
)

status = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "checkpoint_path": str(checkpoint_path) if checkpoint_path else None,
    "n_predictions": int(len(pred)),
    "n_resolved_images": int(pred["resolved_image_path"].notna().sum()),
    "target_mode": os.environ.get("THERMO_GRADCAM_TARGET_MODE", "positive_logit"),
}


# -----------------------------
# Fail-safe empty output if inputs are incomplete
# -----------------------------

if not can_attempt:
    cols = list(pred.columns) + [
        "gradcam_heatmap_path",
        "gradcam_overlay_path",
        "gradcam_max_x",
        "gradcam_max_y",
        "gradcam_image_width",
        "gradcam_image_height",
        "gradcam_target_mode",
        "gradcam_model_probability_recomputed",
        "gradcam_status",
        "gradcam_error",
    ]

    pd.DataFrame(columns=cols).to_csv(
        CONFIG_DIR / "gradcam_summary.csv",
        index=False
    )

    status["n_gradcam_generated"] = 0
    status["failure_reason"] = (
        "No checkpoint found or no image paths resolved. "
        "Check gradcam_path_resolution_report.json and gradcam_auto_unzip_report.json."
    )

else:
    import torch
    import torch.nn as nn
    from torchvision import models, transforms

    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    IMG_SIZE = int(record.get("img_size", os.environ.get("THERMO_CNN_SIZE", 224)))
    MODEL_NAME = record.get("model_name", "resnet18")

    if MODEL_NAME != "resnet18":
        raise ValueError(f"Unsupported model for this notebook: {MODEL_NAME}")

    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, 1)

    ckpt = torch.load(checkpoint_path, map_location=DEVICE)
    state = ckpt.get("model_state_dict", ckpt)

    model.load_state_dict(state)
    model.to(DEVICE)
    model.eval()

    target_layer = model.layer4[-1]
    acts = {}
    grads = {}

    h1 = target_layer.register_forward_hook(
        lambda m, i, o: acts.__setitem__("v", o.detach())
    )

    h2 = target_layer.register_full_backward_hook(
        lambda m, gi, go: grads.__setitem__("v", go[0].detach())
    )

    tfm = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        ),
    ])

    target_mode = os.environ.get("THERMO_GRADCAM_TARGET_MODE", "positive_logit")

    def make_one(image_path, out_prefix, predicted_label):
        img = Image.open(image_path).convert("RGB")
        img = img.resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)

        x = tfm(img).unsqueeze(0).to(DEVICE)

        model.zero_grad(set_to_none=True)

        logit = model(x).squeeze()
        prob = float(torch.sigmoid(logit).detach().cpu())

        if target_mode == "predicted_class":
            target = logit if int(predicted_label) == 1 else -logit
        elif target_mode == "positive_logit":
            target = logit
        else:
            raise ValueError(
                "THERMO_GRADCAM_TARGET_MODE must be 'positive_logit' or 'predicted_class'."
            )

        target.backward()

        A = acts["v"][0]
        G = grads["v"][0]

        w = G.mean(dim=(1, 2))
        cam = torch.relu((w[:, None, None] * A).sum(dim=0))
        cam = cam.detach().cpu().numpy()

        if np.nanmax(cam) > np.nanmin(cam):
            cam = (cam - np.nanmin(cam)) / (np.nanmax(cam) - np.nanmin(cam))
        else:
            cam = np.zeros_like(cam)

        cam = np.array(
            Image.fromarray((cam * 255).astype(np.uint8)).resize(
                (IMG_SIZE, IMG_SIZE),
                Image.BILINEAR
            )
        ) / 255.0

        my, mx = np.unravel_index(np.nanargmax(cam), cam.shape)

        safe_stem = Path(str(out_prefix)).stem.replace(" ", "_")
        hp = GRADCAM_DIR / f"{safe_stem}_heatmap.npy"
        op = GRADCAM_DIR / f"{safe_stem}_overlay.png"

        np.save(hp, cam)

        plt.figure(figsize=(4, 4))
        plt.imshow(img)
        plt.imshow(cam, alpha=0.45)
        plt.axis("off")
        plt.tight_layout()
        plt.savefig(op, dpi=200, bbox_inches="tight", pad_inches=0)
        plt.close()

        return str(hp), str(op), int(mx), int(my), prob


    selected = pred[
        pred["split"].astype(str).str.lower().eq("test")
        & pred["resolved_image_path"].notna()
    ].copy()

    if selected.empty:
        selected = pred[pred["resolved_image_path"].notna()].copy()
        warnings.warn(
            "No rows with split == 'test'. Grad-CAM was generated for all resolved images."
        )

    rows = []

    for _, row in selected.reset_index(drop=True).iterrows():
        prefix = f"{row.get('split', 'unknown')}_{Path(str(row['image_name'])).stem}"

        try:
            hp, op, mx, my, prob = make_one(
                row["resolved_image_path"],
                prefix,
                row.get("cnn_predicted_label_binary", 1)
            )
            st = "ok"
            err = ""

        except Exception as e:
            hp = None
            op = None
            mx = np.nan
            my = np.nan
            prob = np.nan
            st = "failed"
            err = repr(e)

        out = row.to_dict()
        out.update({
            "gradcam_heatmap_path": hp,
            "gradcam_overlay_path": op,
            "gradcam_max_x": mx,
            "gradcam_max_y": my,
            "gradcam_image_width": IMG_SIZE,
            "gradcam_image_height": IMG_SIZE,
            "gradcam_target_mode": target_mode,
            "gradcam_model_probability_recomputed": prob,
            "gradcam_status": st,
            "gradcam_error": err,
        })

        rows.append(out)

    h1.remove()
    h2.remove()

    pd.DataFrame(rows).to_csv(
        CONFIG_DIR / "gradcam_summary.csv",
        index=False
    )

    status["n_gradcam_generated"] = int(
        pd.DataFrame(rows)["gradcam_status"].eq("ok").sum()
    )


# -----------------------------
# Save status
# -----------------------------

(CONFIG_DIR / "gradcam_status.json").write_text(
    json.dumps(status, indent=2),
    encoding="utf-8"
)

(REPORTS_DIR / "gradcam_status.json").write_text(
    json.dumps(status, indent=2),
    encoding="utf-8"
)

print(json.dumps(status, indent=2))

{
  "auto_unzip_executed": true,
  "n_zip_files_found": 2,
  "n_archives_extracted": 2,
  "n_archives_skipped": 0,
  "clean_images_before": 0,
  "clean_images_after": 347,
  "dataset_split_images_before": 0,
  "dataset_split_images_after": 347,
  "extracted_archives": [
    {
      "zip": "/content/clean_images_224x224.zip",
      "to": "/content/project_thermography_equine/data/processed/clean_images",
      "n_image_members": 347
    },
    {
      "zip": "/content/dataset_split-20260605T090743Z-3-001.zip",
      "to": "/content/project_thermography_equine/data/dataset_split",
      "n_image_members": 347
    }
  ],
  "skipped_archives": []
}
{
  "n_predictions": 347,
  "n_resolved_images": 347,
  "n_unresolved_images": 0,
  "n_images_in_clean_image_dir": 347,
  "n_images_in_dataset_split": 347,
  "example_unresolved_image_names": []
}
{
  "created_utc": "2026-06-07T06:51:54.634285+00:00",
  "checkpoint_path": "/content/project_thermography_equine/outputs/models/cnn_resnet18_best.pt"

In [2]:

from pathlib import Path
import shutil
import pandas as pd

summary_path = CONFIG_DIR / "gradcam_summary.csv"

if not summary_path.exists():
    raise FileNotFoundError(
        f"Grad-CAM status indicates generation, but summary file was not found: {summary_path}"
    )

for dst in [
    CONFIG_DIR / "gradcam_summary.csv",
    REPORTS_DIR / "gradcam_summary.csv",
    GRADCAM_DIR / "gradcam_summary.csv",
    OUTPUT_ROOT / "gradcam_summary.csv",
]:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if summary_path.resolve() != dst.resolve():
        shutil.copy2(summary_path, dst)

print("Grad-CAM summary exported to:")
for p in [
    CONFIG_DIR / "gradcam_summary.csv",
    REPORTS_DIR / "gradcam_summary.csv",
    GRADCAM_DIR / "gradcam_summary.csv",
    OUTPUT_ROOT / "gradcam_summary.csv",
]:
    print(p, "exists=", p.exists())

Grad-CAM summary exported to:
/content/project_thermography_equine/outputs/config/gradcam_summary.csv exists= True
/content/project_thermography_equine/outputs/reports/gradcam_summary.csv exists= True
/content/project_thermography_equine/outputs/gradcam/gradcam_summary.csv exists= True
/content/project_thermography_equine/outputs/gradcam_summary.csv exists= True
